In [6]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import re

# Load the dataset
df = pd.read_csv('Smishing_dataset1.csv')

# Clean the text: remove special placeholders like <#> and <DECIMAL>
def clean_text(text):
    text = re.sub(r'[<][#][>]|<DECIMAL>', '', text)
    return text.strip()

df['Message'] = df['Message'].apply(clean_text)

# Convert labels to binary (ham: 0, smish: 1)
df['Labels'] = df['Labels'].map({'ham': 0, 'smish': 1})

# Extract features using TF-IDF
vectorizer = TfidfVectorizer(max_features=500, stop_words='english')
X = vectorizer.fit_transform(df['Message']).toarray()
y = df['Labels'].values

print("Feature matrix shape:", X.shape)
print("Label array shape:", y.shape)

Feature matrix shape: (11743, 500)
Label array shape: (11743,)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity matrix
similarity_matrix = cosine_similarity(X)

# Construct incidence matrix H using k-NN (k=5)
k = 5
num_nodes = X.shape[0]
H = np.zeros((num_nodes, num_nodes))

for i in range(num_nodes):
    similar_indices = np.argsort(similarity_matrix[i])[-(k + 1):-1]
    H[similar_indices, i] = 1.0

# Compute degrees as vectors
Dv_vec = np.sum(H, axis=1)  # Vertex degrees
De_vec = np.sum(H, axis=0)  # Edge degrees

# Convert to diagonal matrices
Dv = np.diag(Dv_vec ** -0.5)
De = np.diag(De_vec ** -1)
Dv[np.isinf(Dv)] = 0
De[np.isinf(De)] = 0

print("Incidence matrix shape:", H.shape)
print("Vertex degree matrix shape:", Dv.shape)
print("Edge degree matrix shape:", De.shape)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8840\908027275.py:20: RuntimeWarning: divide by zero encountered in power
  Dv = np.diag(Dv_vec ** -0.5)


Incidence matrix shape: (11743, 11743)
Vertex degree matrix shape: (11743, 11743)
Edge degree matrix shape: (11743, 11743)


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Convert full matrices to tensors
X_tensor = torch.FloatTensor(X)
H_tensor = torch.FloatTensor(H)
y_tensor = torch.LongTensor(y)
Dv_tensor = torch.FloatTensor(Dv)
De_tensor = torch.FloatTensor(De)

class HGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.5):
        super(HGNN, self).__init__()
        self.conv1 = nn.Linear(input_dim, hidden_dim)
        self.conv2 = nn.Linear(hidden_dim, output_dim)
        self.dropout = dropout

    def forward(self, X, H, Dv, De):
        # Normalize and compute convolution per batch
        batch_size = X.size(0)
        num_nodes = H.size(0)  # Full number of nodes

        # Slice H, Dv, De for the batch (simplified batching)
        H_batch = H[:batch_size, :batch_size]  # Assume batch is a subset
        Dv_batch = Dv[:batch_size, :batch_size]
        De_batch = De[:batch_size, :batch_size]

        # Apply convolution: D_v^{-1/2} H D_e^{-1} H^T D_v^{-1/2} X
        X = torch.matmul(Dv_batch, X)  # (batch_size, input_dim)
        X = torch.matmul(H_batch, torch.matmul(De_batch, torch.matmul(H_batch.T, X)))
        X = F.relu(self.conv1(X))
        X = F.dropout(X, self.dropout, training=self.training)

        # Second layer
        X = torch.matmul(Dv_batch, X)
        X = torch.matmul(H_batch, torch.matmul(De_batch, torch.matmul(H_batch.T, X)))
        X = self.conv2(X)
        return F.log_softmax(X, dim=1)

# Initialize model
model = HGNN(input_dim=X.shape[1], hidden_dim=16, output_dim=2)
print(model)

HGNN(
  (conv1): Linear(in_features=500, out_features=16, bias=True)
  (conv2): Linear(in_features=16, out_features=2, bias=True)
)


In [9]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# Split data (X and y only, H, Dv, De are full)
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42)

# Create data loaders (H, Dv, De are not batched yet)
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Training setup
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

# Training loop
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        # Repeat H, Dv, De for the batch (simplified)
        H_batch = H_tensor[:X_batch.size(0), :X_batch.size(0)]
        Dv_batch = Dv_tensor[:X_batch.size(0), :X_batch.size(0)]
        De_batch = De_tensor[:X_batch.size(0), :X_batch.size(0)]
        output = model(X_batch, H_batch, Dv_batch, De_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}')

model.eval()

Epoch [10/50], Loss: 0.3584
Epoch [20/50], Loss: 0.3579
Epoch [30/50], Loss: 0.3537
Epoch [40/50], Loss: 0.3493
Epoch [50/50], Loss: 0.3430


HGNN(
  (conv1): Linear(in_features=500, out_features=16, bias=True)
  (conv2): Linear(in_features=16, out_features=2, bias=True)
)

In [10]:
from sklearn.metrics import accuracy_score

# Evaluate on test set
model.eval()
with torch.no_grad():
    X_test_tensor = X_test
    H_test_tensor = H_tensor[:X_test.size(0), :X_test.size(0)]
    Dv_test_tensor = Dv_tensor[:X_test.size(0), :X_test.size(0)]
    De_test_tensor = De_tensor[:X_test.size(0), :X_test.size(0)]
    output = model(X_test_tensor, H_test_tensor, Dv_test_tensor, De_test_tensor)
    _, predicted = torch.max(output.data, 1)
    accuracy = accuracy_score(y_test.cpu(), predicted.cpu())
    print(f'Test Accuracy: {accuracy:.4f}')

Test Accuracy: 0.8540
